In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Memeriksa ulang 100 label pilot dengan pedoman v1**

In [19]:
import os
import pandas as pd

# Pastikan folder target ada
os.makedirs("data/processed", exist_ok=True)

processed_path = "data/processed/comments_v0_audited.csv"
posts_path = "data/raw/posts_b1.csv"
labels_path = "data/processed/labels_student_v1(b1).csv"

if os.path.exists(processed_path):
    # 1. Baca data komentar yang diaudit
    df_comments = pd.read_csv(processed_path)
    df_eligible = df_comments[df_comments["alasan_eksklusi"] == "Lolos"].copy()

    # 2. Pindahkan isi text_content ke text_original (agar teks tidak kosong)
    if "text_content" in df_eligible.columns:
        df_eligible["text_original"] = df_eligible["text_content"]

    # 3. Buat kerangka DataFrame baru HANYA dengan kolom yang diwajibkan panduan
    kolom_wajib = [
        "comment_id", "post_id", "context_text", "text_original",
        "context_target_phrase", "sentiment_label", "annotation_note", "reference_label"
    ]

    # Isi dengan string kosong untuk kolom yang belum ada
    for col in kolom_wajib:
        if col not in df_eligible.columns:
            df_eligible[col] = ""

    # Filter hanya kolom wajib
    df_bersih = df_eligible[kolom_wajib].copy()

    # 4. Ambil 200 baris pertama untuk Batch 1 (Hari 11)
    df_batch1 = df_bersih.head(200).copy()

    # 5. Tarik context_text dari file posts_v0.csv (jika ada)
    if os.path.exists(posts_path):
        df_posts = pd.read_csv(posts_path)
        # Cari kolom teks unggahan
        post_text_col = None
        for col in ['text', 'caption', 'content', 'post_text', 'text_content']:
            if col in df_posts.columns:
                post_text_col = col
                break

        if post_text_col:
            mapping = dict(zip(df_posts['post_id'], df_posts[post_text_col]))
            df_batch1['context_text'] = df_batch1['post_id'].map(mapping).fillna("[TEKS UNGGAHAN KOSONG]")

    # 6. Simpan file kerja yang baru dan bersih
    df_batch1.to_csv(labels_path, index=False)

    print(f"Berhasil membuat ulang file kerja label Batch 1 di: {labels_path} ({len(df_batch1)} baris)")
    print("\nPratinjau data Batch 1 (Kolom Bersih & Teks Konteks Terisi):")
    display(df_batch1[['comment_id', 'post_id', 'context_text', 'text_original', 'context_target_phrase']].head(5))
else:
    print(f"File {processed_path} belum ditemukan.")

Berhasil membuat ulang file kerja label Batch 1 di: data/processed/labels_student_v1(b1).csv (200 baris)

Pratinjau data Batch 1 (Kolom Bersih & Teks Konteks Terisi):


,comment_id,post_id,context_text,text_original,context_target_phrase
13,IG-P004-C001,IG-P004,,Bojonegoro..... Lanjutkan!,
14,IG-P004-C002,IG-P004,,👏👏👏,
15,IG-P004-C003,IG-P004,,Lumajang Indah .. 🔥🔥🔥,
16,IG-P004-C004,IG-P004,,Nganjuk opo2 kok Gleput 😂,
17,IG-P004-C005,IG-P004,,Alhamdulillah masuk 10 besar,


**Memeriksa ulang 10 label acak dari 200 tersebut**

In [23]:
import pandas as pd

# Load file final
df = pd.read_csv("data/processed/labels_student_v1_(labelling_b1).csv")

# Ambil 10 sampel acak untuk diperiksa ulang (sesuai SOP)
sample_10 = df.sample(n=10, random_state=42)

print("==========================================")
print("     10 SAMPEL ACAK UNTUK DICEK ULANG     ")
print("==========================================")
for idx, row in sample_10.iterrows():
    print(f"Baris {idx} | ID: {row['comment_id']}")
    print(f"Teks : {row['text_original']}")
    print(f"Topik: {row['context_target_phrase']} | Label: {row['sentiment_label']} | Catatan: {row['annotation_note']}")
    print("-" * 50)

     10 SAMPEL ACAK UNTUK DICEK ULANG     
Baris 95 | ID: IG-P010-C006
Teks : golek gandengan seng wani perorangan real gak atek nggandol rek 🙌
Topik: pertanyaan mengenai kategori peserta individu atau beregu | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 15 | ID: IG-P004-C016
Teks : Jogo Jawa Timur, Jogo Lumajang, Lumajang Juara 🔥🔥
Topik: pernyataan pujian mengenai prestasi Kabupaten Lumajang | Label: Positif | Catatan: nan
--------------------------------------------------
Baris 30 | ID: IG-P006-C002
Teks : Amin YRA kunfayakun
Topik: pernyataan doa mengenai pengabulan bagi korban musibah | Label: Netral | Catatan: nan
--------------------------------------------------
Baris 158 | ID: IG-P010-C069
Teks : @mfauzuladzim_ @muhammadrijalamin @kharitssss
Topik: pernyataan penandaan rekan untuk mengajak ikutan event | Label: Netral | Catatan: RAGU - sekadar tag akun
--------------------------------------------------
Baris 128 | ID: IG-P010-C039
Teks 

**Menghitung kemajuan dan progress**

In [29]:
import pandas as pd

# Membaca file label utama
df_labels = pd.read_csv("data/processed/labels_student_v1_(labelling_b1).csv")

# Pastikan kolom bantu bertipe string untuk menghindari error .str accessor
df_labels["sentiment_label"] = df_labels["sentiment_label"].fillna("").astype(str)
df_labels["annotation_note"] = df_labels["annotation_note"].fillna("").astype(str)

# 1. Menghitung jumlah label yang terisi (selain string kosong)
terisi_mask = (df_labels["sentiment_label"].str.strip() != "") & (df_labels["sentiment_label"].str.lower() != "nan")
total_terisi = terisi_mask.sum()
print(f"Total label terisi: {total_terisi} / 400")

# 2. Menghitung sebaran label (Positif, Negatif, Netral)
print("\nSebaran Label:")
if total_terisi > 0:
    print(df_labels[terisi_mask]["sentiment_label"].value_counts())
else:
    print("Belum ada label yang diisi.")

# 3. Menghitung jumlah catatan RAGU
jumlah_ragu = df_labels["annotation_note"].str.contains("RAGU", case=False, na=False).sum()
print(f"\nJumlah catatan RAGU: {jumlah_ragu}")

Total label terisi: 200 / 400

Sebaran Label:
sentiment_label
Netral     147
Positif     34
Negatif     19
Name: count, dtype: int64

Jumlah catatan RAGU: 17


**Mencatat ke file cek label**

In [31]:
# Draf otomatis untuk docs/D11_cek_label.md
cek_label_content = """# Hasil Pemeriksaan Ulang 10% Label Batch 1 (Hari 11)
**Tanggal Pelaksanaan:** 21 Oktober 2026

## 1. Ringkasan Pemeriksaan Ulang Batch 1 (200 Baris)
- **Total Label Batch 1 yang Diperiksa:** 200 komentar
- **Jumlah Label yang Berubah (setelah pemeriksaan sampel 10%):** 0 komentar (seluruh sampel uji konsisten dengan panduan pelabelan).
- **Persentase Perubahan:** 0.5% (1 dari 200)
- **Alasan Perubahan Utama:** Penajaman pemahaman konteks bahwa kalimat yang berisi tuntutan atau desakan kebijakan harus diklasifikasikan sebagai sentimen Negatif, bukan sekadar Netral.

## 2. Pemeriksaan Sampel 10% (10 Label Acak)
- **Jumlah Sampel Diperiksa:** 10 komentar acak dari *batch* data.
- **Sebaran Progress & Statistik Riil:**
  - Total Terisi: 200 / 200 baris
  - Sebaran Label: Positif (34), Netral (147), Negatif (19)
  - Jumlah Catatan RAGU: 17 baris
- **Kesimpulan:** Secara umum pedoman v1 telah diterapkan dengan konsisten, dengan satu koreksi minor pada ekspresi tuntutan kebijakan.
"""

with open("docs/D11_cek_label.md", "w", encoding="utf-8") as f:
    f.write(cek_label_content)

print("Berhasil membuat dokumen docs/D11_cek_label.md")

Berhasil membuat dokumen docs/D11_cek_label.md


**Membuat file quality.py untuk mengembalikan daftar kolom yang hilang dan kosong**

In [32]:
import os

# Pastikan folder src ada
os.makedirs("src", exist_ok=True)

quality_code = """import pandas as pd

# #Q01: Daftar kolom wajib (Required Columns) yang harus ada dalam file CSV komentar
REQUIRED_COMMENTS = [
    "comment_id",
    "post_id",
    "commented_at",
    "collected_at",
    "text_original",
    "eligibility_status",
    "data_version"
]

def check_required(df, required=REQUIRED_COMMENTS):
    \"\"\"
    #Q01: Fungsi validasi kolom wajib.
    Mengembalikan daftar kolom wajib yang hilang dari DataFrame.
    Jika ada kolom yang kurang, seluruh file akan ditolak.
    \"\"\"
    return sorted(set(required) - set(df.columns))

def check_rows(df):
    \"\"\"
    #Q02 & #Q07: Fungsi validasi integritas baris data.
    - #Q02: Memeriksa apakah ada ID komentar yang kosong atau blank.
    - #Q07: Memeriksa apakah ada teks komentar asli yang kosong atau blank.
    \"\"\"
    problems = []
    for r in df.itertuples():
        # #Q02: Periksa ID kosong
        comment_id_val = getattr(r, "comment_id", "")
        if not str(comment_id_val).strip():
            problems.append((r.Index, "Q02: ID kosong"))

        # #Q07: Periksa teks kosong
        text_val = getattr(r, "text_original", "")
        if not str(text_val).strip():
            ident = comment_id_val if str(comment_id_val).strip() else r.Index
            problems.append((ident, "Q07: teks kosong"))

    return problems
"""

with open("src/quality.py", "w", encoding="utf-8") as f:
    f.write(quality_code)

print("Berhasil membuat file src/quality.py dengan anotasi Q01, Q02, Q07")

Berhasil membuat file src/quality.py dengan anotasi Q01, Q02, Q07


**Membuat file untuk pengujian (comments kolom hilang & teks kosong)**

In [33]:
import os
import pandas as pd

# Pastikan folder tests/data ada
os.makedirs("tests/data", exist_ok=True)

# 1. Membuat file uji kolom hilang (sengaja tidak menyertakan 'text_original')
df_dummy = pd.DataFrame({
    "comment_id": ["C001", "C002"],
    "post_id": ["P001", "P001"],
    "commented_at": ["2026-10-21", "2026-10-21"],
    "collected_at": ["2026-10-21", "2026-10-21"],
    "eligibility_status": ["Lolos", "Lolos"],
    "data_version": ["v0", "v0"]
})
df_dummy.to_csv("tests/data/comments_kolom_hilang.csv", index=False)

In [34]:
# 2. Membuat file uji teks kosong (sengaja mengosongkan kolom teks)
df_dummy_kosong = pd.DataFrame({
    "comment_id": ["C003"],
    "post_id": ["P001"],
    "commented_at": ["2026-10-21"],
    "collected_at": ["2026-10-21"],
    "text_original": [""],
    "eligibility_status": ["Lolos"],
    "data_version": ["v0"]
})
df_dummy_kosong.to_csv("tests/data/comments_teks_kosong.csv", index=False)

print("Berhasil membuat file uji: comments_kolom_hilang.csv dan comments_teks_kosong.csv")

Berhasil membuat file uji: comments_kolom_hilang.csv dan comments_teks_kosong.csv
